# CLIP Fine-Tuning & Embedding Generation — Offline Indexing
**Role in pipeline:** Step 3 of offline indexing.  
CLIP fuses cropped image embeddings with BLIP-2 caption embeddings into a single normalized vector per product.

**Ablation conditions:**
- **A** — Vision-only CLIP (α=1), frozen, no fine-tuning. Baseline.
- **B** — Frozen CLIP + frozen BLIP-2, fused embeddings (two α values).
- **C** — Fine-tuned CLIP + frozen BLIP-2, fused embeddings (two α values).

**Inputs:**
- `captions.json` from BLIP notebook
- Raw catalog images + bbox annotations
- Fine-tuned YOLO `.pt` for online query crops

**Outputs:**
- `clip_finetuned.pt` — fine-tuned CLIP vision encoder weights
- `gallery_index_<condition>.bin` — HNSW index per ablation condition
- `gallery_meta_<condition>.json` — metadata (item_id, image_name) per condition
- `metrics_<condition>.json` — Recall@K, NDCG@K, mAP@K results

## Cell 1 — Install Dependencies

In [1]:
!pip install open-clip-torch hnswlib tqdm --quiet
!pip install ultralytics --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.6 MB/s eta 0:00:00


## Cell 2 — Imports & Paths

In [2]:
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import open_clip
import hnswlib
from collections import defaultdict

# ── Paths (same as BLIP notebook) ──
DATASET_ROOT  = Path("/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset")
IMG_ROOT      = DATASET_ROOT / "img" / "img"
SPLIT_FILE    = DATASET_ROOT / "eval" / "list_eval_partition.txt"
BBOX_FILE     = DATASET_ROOT / "Anno" / "list_bbox_inshop.txt"

# ── Input from BLIP notebook ──
# Upload captions.json as a Kaggle dataset and point here:
CAPTIONS_FILE = Path("/kaggle/input/datasets/taralsanka/blip-updated-captions/captions.json")  # adjust dataset name if needed

# ── YOLO model (same as used in BLIP notebook's online_crop) ──
YOLO_PT = Path("/kaggle/input/datasets/taralsanka/best-yolo-pt/best_yolo8l.pt")

# ── Output ──
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Config ──
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MODEL = "ViT-L-14"         # strong backbone; use ViT-B-32 if GPU OOM
CLIP_PRETRAIN = "openai"        # openai pretrained weights
EMB_DIM    = 768                # ViT-L-14 embedding dim; change to 512 for ViT-B-32
PAD        = 0.05               # bbox padding fraction
BATCH_SIZE = 64                 # for embedding generation; reduce if OOM
TOP_K_LIST = [5, 10, 15]        # evaluation K values

# ── Seeds (use roll numbers for reproducibility) ──
SEEDS = [83, 588, 527, 33]     # replace with team roll numbers

print(f"Device       : {DEVICE}")
print(f"CLIP model   : {CLIP_MODEL}")
print(f"IMG_ROOT     : {IMG_ROOT.exists()}")
print(f"Captions file: {CAPTIONS_FILE.exists()}")
print(f"YOLO pt      : {YOLO_PT.exists()}")

Device       : cuda
CLIP model   : ViT-L-14
IMG_ROOT     : True
Captions file: True
YOLO pt      : True


## Cell 3 — Seed & Annotation Parsing Helpers

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parse_split_file(path):
    """Returns list of dicts: {image_name, item_id, split}."""
    with open(path) as f:
        lines = f.readlines()
    rows = []
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        raw = parts[0]
        img_name = raw[len("img/"):] if raw.startswith("img/") else raw
        rows.append({"image_name": img_name, "item_id": parts[1], "split": parts[2]})
    return rows


def parse_bbox_file(path):
    """Returns dict: image_name -> (x1, y1, x2, y2)."""
    with open(path) as f:
        lines = f.readlines()
    bboxes = {}
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        raw = parts[0]
        img_name = raw[len("img/"):] if raw.startswith("img/") else raw
        bboxes[img_name] = (int(parts[3]), int(parts[4]), int(parts[5]), int(parts[6]))
    return bboxes


def bbox_crop(img_path, bbox, pad=PAD):
    """Crop image using ground-truth bbox annotation (offline indexing)."""
    pil_img = Image.open(img_path).convert("RGB")
    W, H = pil_img.size
    x1, y1, x2, y2 = bbox
    px = int((x2 - x1) * pad)
    py = int((y2 - y1) * pad)
    x1 = max(0, x1 - px); y1 = max(0, y1 - py)
    x2 = min(W, x2 + px); y2 = min(H, y2 + py)
    if x2 <= x1 or y2 <= y1:
        return pil_img
    return pil_img.crop((x1, y1, x2, y2))


# Load all annotations once
all_rows  = parse_split_file(SPLIT_FILE)
bbox_map  = parse_bbox_file(BBOX_FILE)
captions  = json.load(open(CAPTIONS_FILE))

train_rows   = [r for r in all_rows if r["split"] == "train"]
gallery_rows = [r for r in all_rows if r["split"] == "gallery"]
query_rows   = [r for r in all_rows if r["split"] == "query"]

print(f"Train   : {len(train_rows)}")
print(f"Gallery : {len(gallery_rows)}")
print(f"Query   : {len(query_rows)}")
print(f"Captions: {len(captions)}")

Train   : 25882
Gallery : 12612
Query   : 14218
Captions: 38494


## Cell 4 — Load CLIP Model

In [4]:
# open_clip gives us easy access to frozen text encoder and trainable vision encoder
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAIN
)
tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
clip_model = clip_model.to(DEVICE)
clip_model.eval()

print(f"CLIP model loaded: {CLIP_MODEL}")
print(f"Vision encoder params: {sum(p.numel() for p in clip_model.visual.parameters()):,}")

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP model loaded: ViT-L-14
Vision encoder params: 303,966,208


## Cell 5 — CLIP Fine-Tuning Dataset
Build (anchor, positive) pairs from the train split.  
Two images sharing the same `item_id` form a positive pair.  
We use **InfoNCE contrastive loss** (same as original CLIP training) over image-only pairs.

In [5]:
class FashionPairDataset(Dataset):
    """
    For each __getitem__, return (anchor_crop, positive_crop) — two different
    images of the same item_id. Used for contrastive fine-tuning of CLIP.
    """
    def __init__(self, rows, bbox_map, img_root, transform, pad=PAD):
        # Group by item_id; keep only items with >=2 images
        groups = defaultdict(list)
        for r in rows:
            groups[r["item_id"]].append(r["image_name"])
        self.items = [(iid, imgs) for iid, imgs in groups.items() if len(imgs) >= 2]
        self.bbox_map  = bbox_map
        self.img_root  = img_root
        self.transform = transform
        self.pad       = pad

    def __len__(self):
        return len(self.items)

    def _load_crop(self, img_name):
        img_path = self.img_root / img_name
        bbox = self.bbox_map.get(img_name)
        if bbox:
            crop = bbox_crop(img_path, bbox, self.pad)
        else:
            crop = Image.open(img_path).convert("RGB")
        return self.transform(crop)

    def __getitem__(self, idx):
        item_id, imgs = self.items[idx]
        anchor_name, pos_name = random.sample(imgs, 2)
        try:
            anchor = self._load_crop(anchor_name)
            positive = self._load_crop(pos_name)
        except Exception:
            # fallback: repeat same item with different pair
            anchor = self._load_crop(anchor_name)
            positive = anchor
        return anchor, positive


train_dataset = FashionPairDataset(train_rows, bbox_map, IMG_ROOT, clip_preprocess)
print(f"Train pairs (unique items): {len(train_dataset)}")

Train pairs (unique items): 3985


## Cell 6 — CLIP Fine-Tuning
**What we freeze / unfreeze:**
- CLIP text encoder → **fully frozen**
- CLIP vision encoder → **freeze all, then unfreeze last 4 transformer blocks**

**Loss:** InfoNCE (symmetric cross-entropy over cosine similarities in a batch).

In [6]:
# ── Fine-tuning hyperparameters ──
FT_EPOCHS     = 5
FT_BATCH_SIZE = 32    # reduce to 16 if GPU OOM
FT_LR         = 1e-5
FT_SEED       = SEEDS[0]  # use first seed for fine-tuning run
LAST_N_BLOCKS = 4          # unfreeze last N vision transformer blocks
TEMPERATURE   = 0.07       # InfoNCE temperature

set_seed(FT_SEED)

# ── Freeze everything first ──
for p in clip_model.parameters():
    p.requires_grad = False

# ── Unfreeze last LAST_N_BLOCKS of vision transformer ──
# open_clip ViT stores blocks in clip_model.visual.transformer.resblocks
vision_blocks = list(clip_model.visual.transformer.resblocks)
for block in vision_blocks[-LAST_N_BLOCKS:]:
    for p in block.parameters():
        p.requires_grad = True

# Also unfreeze vision projection (ln_post, proj)
for p in clip_model.visual.ln_post.parameters():
    p.requires_grad = True
if hasattr(clip_model.visual, 'proj') and clip_model.visual.proj is not None:
    clip_model.visual.proj.requires_grad = True

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in clip_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


def infonce_loss(z1, z2, temperature=TEMPERATURE):
    """
    Symmetric InfoNCE (NT-Xent) loss.
    z1, z2: (B, D) L2-normalized embeddings.
    Diagonal = positive pairs; off-diagonal = negatives within batch.
    """
    B = z1.shape[0]
    # All 2B embeddings concatenated
    z = torch.cat([z1, z2], dim=0)                      # (2B, D)
    sim = (z @ z.T) / temperature                       # (2B, 2B)
    # Mask out self-similarity
    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, float('-inf'))
    # Positive pair indices: z1[i] <-> z2[i] => index B+i; z2[i] <-> z1[i] => index i
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)
    loss = F.cross_entropy(sim, labels)
    return loss


ft_loader = DataLoader(
    train_dataset,
    batch_size=FT_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,    # InfoNCE needs full batches
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, clip_model.parameters()),
    lr=FT_LR, weight_decay=0.01
)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

clip_model.train()

for epoch in range(FT_EPOCHS):
    total_loss = 0.0
    for anchor, positive in tqdm(ft_loader, desc=f"Epoch {epoch+1}/{FT_EPOCHS}"):
        anchor   = anchor.to(DEVICE)
        positive = positive.to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            z1 = F.normalize(clip_model.encode_image(anchor),   dim=-1)
            z2 = F.normalize(clip_model.encode_image(positive), dim=-1)
            loss = infonce_loss(z1, z2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, clip_model.parameters()), 1.0
        )
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg = total_loss / len(ft_loader)
    print(f"  Epoch {epoch+1} — avg InfoNCE loss: {avg:.4f}")

# Save fine-tuned vision encoder
ft_path = OUTPUT_DIR / "clip_finetuned.pt"
torch.save(clip_model.state_dict(), ft_path)
print(f"\nSaved fine-tuned CLIP → {ft_path}")

/tmp/ipykernel_23/1016203328.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


Trainable params: 51,173,376 / 427,616,513 (12.0%)


Epoch 1/5:   0%|          | 0/124 [00:00<?, ?it/s]/tmp/ipykernel_23/1016203328.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
Epoch 1/5: 100%|██████████| 124/124 [02:15<00:00,  1.10s/it]


  Epoch 1 — avg InfoNCE loss: 1.0812


Epoch 2/5: 100%|██████████| 124/124 [02:24<00:00,  1.17s/it]


...

  Epoch 3 — avg InfoNCE loss: 0.5059


Epoch 4/5: 100%|██████████| 124/124 [02:24<00:00,  1.17s/it]


  Epoch 4 — avg InfoNCE loss: 0.4134


Epoch 5/5: 100%|██████████| 124/124 [02:24<00:00,  1.17s/it]


  Epoch 5 — avg InfoNCE loss: 0.3652

Saved fine-tuned CLIP → /kaggle/working/clip_finetuned.pt


## Cell 7 — Embedding Generation Helper
Given a list of image rows and their captions, compute fused CLIP embeddings:

$$v_i = \alpha \cdot \phi_V(\hat{x}_i) + (1-\alpha) \cdot \phi_T(c_i), \quad \|v_i\|=1$$

In [7]:
@torch.no_grad()
def generate_embeddings(rows, captions, clip_model, tokenizer, clip_preprocess,
                        bbox_map, img_root, alpha, batch_size=BATCH_SIZE, pad=PAD):
    """
    Compute fused CLIP embeddings for a list of image rows.

    Returns:
        embeddings : np.ndarray (N, EMB_DIM) — L2-normalized fused vectors
        item_ids   : list[str]  — item_id per row (same order)
        img_names  : list[str]  — image_name per row (same order)
    """
    clip_model.eval()
    all_embs, all_ids, all_names = [], [], []

    for i in tqdm(range(0, len(rows), batch_size), desc=f"Embedding (α={alpha})"):
        batch = rows[i : i + batch_size]
        crops, texts, ids, names = [], [], [], []

        for r in batch:
            img_path = img_root / r["image_name"]
            if not img_path.exists():
                continue
            try:
                bbox = bbox_map.get(r["image_name"])
                crop = bbox_crop(img_path, bbox) if bbox else Image.open(img_path).convert("RGB")
                crops.append(clip_preprocess(crop))
                texts.append(captions.get(r["image_name"], ""))
                ids.append(r["item_id"])
                names.append(r["image_name"])
            except Exception:
                continue

        if not crops:
            continue

        img_tensor  = torch.stack(crops).to(DEVICE)
        txt_tokens  = tokenizer(texts).to(DEVICE)

        img_emb = F.normalize(clip_model.encode_image(img_tensor).float(), dim=-1)
        txt_emb = F.normalize(clip_model.encode_text(txt_tokens).float(),  dim=-1)

        if alpha == 1.0:
            fused = img_emb
        elif alpha == 0.0:
            fused = txt_emb
        else:
            fused = alpha * img_emb + (1.0 - alpha) * txt_emb

        fused = F.normalize(fused, dim=-1)
        all_embs.append(fused.cpu().numpy())
        all_ids.extend(ids)
        all_names.extend(names)

    embeddings = np.vstack(all_embs).astype(np.float32)
    return embeddings, all_ids, all_names


print("Embedding helper defined.")

Embedding helper defined.


## Cell 8 — Build HNSW Index Helper

In [8]:
def build_hnsw_index(embeddings, dim, ef_construction=200, M=32):
    """
    Build an HNSW index over the gallery embeddings.
    Returns the fitted hnswlib index.
    ef_construction / M control index quality vs build speed.
    """
    index = hnswlib.Index(space='cosine', dim=dim)
    index.init_index(max_elements=len(embeddings), ef_construction=ef_construction, M=M)
    index.add_items(embeddings, list(range(len(embeddings))))
    index.set_ef(150)   # query-time ef (higher = better recall, slower)
    return index


print("HNSW index helper defined.")

HNSW index helper defined.


## Cell 9 — Evaluation Metrics
Recall@K, NDCG@K, mAP@K — all computed per query then averaged.

In [9]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    """1 if any of top-k retrieved items is relevant, else 0."""
    top_k = retrieved_ids[:k]
    return int(any(iid in relevant_ids for iid in top_k))


def ndcg_at_k(retrieved_ids, relevant_ids, k):
    """NDCG@K — position-weighted relevance."""
    top_k = retrieved_ids[:k]
    dcg = sum(
        (1.0 / np.log2(rank + 2))  # rank 0-indexed, log2(rank+2) = log2(1+1), log2(2+1)...
        for rank, iid in enumerate(top_k)
        if iid in relevant_ids
    )
    # Ideal DCG: first min(|rel|, k) positions are all relevant
    n_rel = min(len(relevant_ids), k)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(n_rel))
    return dcg / idcg if idcg > 0 else 0.0


def average_precision_at_k(retrieved_ids, relevant_ids, k):
    """AP@K for a single query."""
    top_k = retrieved_ids[:k]
    hits, prec_sum = 0, 0.0
    for rank, iid in enumerate(top_k, 1):
        if iid in relevant_ids:
            hits += 1
            prec_sum += hits / rank
    denom = min(len(relevant_ids), k)
    return prec_sum / denom if denom > 0 else 0.0


def evaluate_retrieval(query_embs, query_ids, gallery_embs, gallery_ids,
                       index, k_list=TOP_K_LIST):
    """
    Run retrieval for all queries and compute Recall@K, NDCG@K, mAP@K.

    Returns dict: {'Recall@5': ..., 'NDCG@5': ..., 'mAP@5': ..., ...}
    """
    max_k = max(k_list)
    per_query = {f"{m}@{k}": [] for m in ["Recall", "NDCG", "mAP"] for k in k_list}

    # Build item_id -> set of gallery positions lookup
    gallery_id_arr = np.array(gallery_ids)

    labels, distances = index.knn_query(query_embs, k=max_k + 1)  # +1 in case query itself is in gallery

    for q_idx, (q_id, q_item) in enumerate(zip(query_ids, query_embs.tolist())):
        retrieved_gallery_pos = labels[q_idx]          # positions in gallery
        retrieved_item_ids    = gallery_id_arr[retrieved_gallery_pos].tolist()

        # Ground truth: gallery images with same item_id (exclude query itself if in gallery)
        relevant_set = set(gallery_id_arr[gallery_id_arr == q_id])  # always {q_id} if matched
        # relevant_set is just the item_id — we compare retrieved item_ids against q_id
        relevant_ids = {q_id}

        for k in k_list:
            per_query[f"Recall@{k}"].append(recall_at_k(retrieved_item_ids, relevant_ids, k))
            per_query[f"NDCG@{k}"].append(ndcg_at_k(retrieved_item_ids, relevant_ids, k))
            per_query[f"mAP@{k}"].append(average_precision_at_k(retrieved_item_ids, relevant_ids, k))

    return {metric: float(np.mean(vals)) for metric, vals in per_query.items()}


print("Evaluation metrics defined.")

Evaluation metrics defined.


## Cell 10 — Ablation A: Vision-Only Frozen CLIP (α = 1)
**Baseline** — no caption fusion, no fine-tuning.

In [10]:
# Reload original pretrained CLIP (no fine-tuning)
clip_frozen, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAIN
)
clip_frozen = clip_frozen.to(DEVICE)
clip_frozen.eval()

all_results = {}  # will collect results from all ablation conditions

# Run over multiple seeds and average
seed_results_A = []
for seed in SEEDS:
    set_seed(seed)

    # Gallery embeddings
    gal_embs, gal_ids, gal_names = generate_embeddings(
        gallery_rows, captions, clip_frozen, tokenizer, clip_preprocess,
        bbox_map, IMG_ROOT, alpha=1.0
    )

    # Query embeddings
    qry_embs, qry_ids, qry_names = generate_embeddings(
        query_rows, captions, clip_frozen, tokenizer, clip_preprocess,
        bbox_map, IMG_ROOT, alpha=1.0
    )

    # Build HNSW
    index_A = build_hnsw_index(gal_embs, dim=EMB_DIM)

    # Evaluate
    metrics = evaluate_retrieval(qry_embs, qry_ids, gal_embs, gal_ids, index_A)
    seed_results_A.append(metrics)
    print(f"Seed {seed}: Recall@10={metrics['Recall@10']:.4f}  mAP@10={metrics['mAP@10']:.4f}")

# Mean ± std across seeds
result_A = {}
for metric in seed_results_A[0]:
    vals = [r[metric] for r in seed_results_A]
    result_A[metric] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}

all_results["A_vision_only_alpha1"] = result_A

print("\n=== Ablation A Results ===")
for k in TOP_K_LIST:
    for m in ["Recall", "NDCG", "mAP"]:
        key = f"{m}@{k}"
        print(f"  {key}: {result_A[key]['mean']:.4f} ± {result_A[key]['std']:.4f}")

# Save index for reuse (last seed's index)
index_A.save_index(str(OUTPUT_DIR / "gallery_index_A.bin"))
json.dump({"item_ids": gal_ids, "img_names": gal_names},
          open(OUTPUT_DIR / "gallery_meta_A.json", "w"), indent=2)
print("\nSaved index A")

Embedding (α=1.0): 100%|██████████| 223/223 [13:49<00:00,  3.72s/it]


Seed 83: Recall@10=0.5771  mAP@10=0.7084


Embedding (α=1.0): 100%|██████████| 223/223 [12:59<00:00,  3.50s/it]


Seed 588: Recall@10=0.5771  mAP@10=0.7084


Embedding (α=1.0): 100%|██████████| 223/223 [12:49<00:00,  3.45s/it]


Seed 527: Recall@10=0.5770  mAP@10=0.7083


Embedding (α=1.0): 100%|██████████| 223/223 [12:54<00:00,  3.47s/it]


Seed 33: Recall@10=0.5770  mAP@10=0.7083

=== Ablation A Results ===
  Recall@5: 0.5132 ± 0.0000
  NDCG@5: 0.5750 ± 0.0000
  mAP@5: 0.6287 ± 0.0000
  Recall@10: 0.5771 ± 0.0000
  NDCG@10: 0.6489 ± 0.0000
  mAP@10: 0.7083 ± 0.0000
  Recall@15: 0.6110 ± 0.0000
  NDCG@15: 0.6890 ± 0.0000
  mAP@15: 0.7473 ± 0.0000

Saved index A


## Cell 11 — Ablation B: Frozen CLIP + Frozen BLIP-2 (two α values)
Measures gain from caption fusion without any fine-tuning.

In [11]:
# Two alpha values for ablation B (choose any two in [0,1])
ALPHA_VALUES_B = [0.7, 0.5]

for alpha in ALPHA_VALUES_B:
    seed_results_B = []
    for seed in SEEDS:
        set_seed(seed)

        gal_embs, gal_ids, gal_names = generate_embeddings(
            gallery_rows, captions, clip_frozen, tokenizer, clip_preprocess,
            bbox_map, IMG_ROOT, alpha=alpha
        )
        qry_embs, qry_ids, qry_names = generate_embeddings(
            query_rows, captions, clip_frozen, tokenizer, clip_preprocess,
            bbox_map, IMG_ROOT, alpha=alpha
        )

        index_B = build_hnsw_index(gal_embs, dim=EMB_DIM)
        metrics = evaluate_retrieval(qry_embs, qry_ids, gal_embs, gal_ids, index_B)
        seed_results_B.append(metrics)
        print(f"  [α={alpha}] Seed {seed}: Recall@10={metrics['Recall@10']:.4f}  mAP@10={metrics['mAP@10']:.4f}")

    result_B = {}
    for metric in seed_results_B[0]:
        vals = [r[metric] for r in seed_results_B]
        result_B[metric] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}

    key_name = f"B_frozen_clip_alpha{str(alpha).replace('.','')}"
    all_results[key_name] = result_B

    print(f"\n=== Ablation B (α={alpha}) Results ===")
    for k in TOP_K_LIST:
        for m in ["Recall", "NDCG", "mAP"]:
            mk = f"{m}@{k}"
            print(f"  {mk}: {result_B[mk]['mean']:.4f} ± {result_B[mk]['std']:.4f}")

    # Save last seed's index
    index_B.save_index(str(OUTPUT_DIR / f"gallery_index_B_alpha{str(alpha).replace('.','')}.bin"))
    json.dump({"item_ids": gal_ids, "img_names": gal_names},
              open(OUTPUT_DIR / f"gallery_meta_B_alpha{str(alpha).replace('.','')}.json", "w"), indent=2)
    print(f"Saved index B α={alpha}")

Embedding (α=0.7): 100%|██████████| 223/223 [12:50<00:00,  3.46s/it]


  [α=0.7] Seed 83: Recall@10=0.4096  mAP@10=0.3732


Embedding (α=0.7): 100%|██████████| 223/223 [12:51<00:00,  3.46s/it]


  [α=0.7] Seed 588: Recall@10=0.4096  mAP@10=0.3731


Embedding (α=0.7): 100%|██████████| 223/223 [12:51<00:00,  3.46s/it]


...

  [α=0.5] Seed 588: Recall@10=0.0741  mAP@10=0.0213


Embedding (α=0.5): 100%|██████████| 223/223 [12:54<00:00,  3.48s/it]


  [α=0.5] Seed 527: Recall@10=0.0740  mAP@10=0.0213


Embedding (α=0.5): 100%|██████████| 223/223 [12:56<00:00,  3.48s/it]


  [α=0.5] Seed 33: Recall@10=0.0740  mAP@10=0.0213

=== Ablation B (α=0.5) Results ===
  Recall@5: 0.0148 ± 0.0000
  NDCG@5: 0.0103 ± 0.0000
  mAP@5: 0.0089 ± 0.0000
  Recall@10: 0.0740 ± 0.0001
  NDCG@10: 0.0347 ± 0.0000
  mAP@10: 0.0213 ± 0.0000
  Recall@15: 0.1000 ± 0.0001
  NDCG@15: 0.0483 ± 0.0000
  mAP@15: 0.0287 ± 0.0000
Saved index B α=0.5


## Cell 12 — Ablation C: Fine-Tuned CLIP + Frozen BLIP-2 (two α values)
Load the fine-tuned CLIP weights from Cell 6.

In [12]:
# Load fine-tuned CLIP
clip_ft, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAIN)
clip_ft.load_state_dict(torch.load(OUTPUT_DIR / "clip_finetuned.pt", map_location="cpu"))
clip_ft = clip_ft.to(DEVICE)
clip_ft.eval()
print("Fine-tuned CLIP loaded.")

# Two alpha values for ablation C (same range as B for fair comparison)
ALPHA_VALUES_C = [0.7, 0.5]

for alpha in ALPHA_VALUES_C:
    seed_results_C = []
    for seed in SEEDS:
        set_seed(seed)

        gal_embs, gal_ids, gal_names = generate_embeddings(
            gallery_rows, captions, clip_ft, tokenizer, clip_preprocess,
            bbox_map, IMG_ROOT, alpha=alpha
        )
        qry_embs, qry_ids, qry_names = generate_embeddings(
            query_rows, captions, clip_ft, tokenizer, clip_preprocess,
            bbox_map, IMG_ROOT, alpha=alpha
        )

        index_C = build_hnsw_index(gal_embs, dim=EMB_DIM)
        metrics = evaluate_retrieval(qry_embs, qry_ids, gal_embs, gal_ids, index_C)
        seed_results_C.append(metrics)
        print(f"  [α={alpha}] Seed {seed}: Recall@10={metrics['Recall@10']:.4f}  mAP@10={metrics['mAP@10']:.4f}")

    result_C = {}
    for metric in seed_results_C[0]:
        vals = [r[metric] for r in seed_results_C]
        result_C[metric] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}

    key_name = f"C_finetuned_clip_alpha{str(alpha).replace('.','')}"
    all_results[key_name] = result_C

    print(f"\n=== Ablation C (α={alpha}) Results ===")
    for k in TOP_K_LIST:
        for m in ["Recall", "NDCG", "mAP"]:
            mk = f"{m}@{k}"
            print(f"  {mk}: {result_C[mk]['mean']:.4f} ± {result_C[mk]['std']:.4f}")

    # Save best index (for Streamlit demo)
    index_C.save_index(str(OUTPUT_DIR / f"gallery_index_C_alpha{str(alpha).replace('.','')}.bin"))
    json.dump({"item_ids": gal_ids, "img_names": gal_names},
              open(OUTPUT_DIR / f"gallery_meta_C_alpha{str(alpha).replace('.','')}.json", "w"), indent=2)
    print(f"Saved index C α={alpha}")

Fine-tuned CLIP loaded.


Embedding (α=0.7): 100%|██████████| 223/223 [12:58<00:00,  3.49s/it]


  [α=0.7] Seed 83: Recall@10=0.8775  mAP@10=1.6674


Embedding (α=0.7): 100%|██████████| 223/223 [12:58<00:00,  3.49s/it]


  [α=0.7] Seed 588: Recall@10=0.8775  mAP@10=1.6674


...

  [α=0.5] Seed 588: Recall@10=0.7894  mAP@10=1.1147


Embedding (α=0.5): 100%|██████████| 223/223 [13:00<00:00,  3.50s/it]


  [α=0.5] Seed 527: Recall@10=0.7894  mAP@10=1.1148


Embedding (α=0.5): 100%|██████████| 223/223 [13:02<00:00,  3.51s/it]


  [α=0.5] Seed 33: Recall@10=0.7894  mAP@10=1.1148

=== Ablation C (α=0.5) Results ===
  Recall@5: 0.7122 ± 0.0001
  NDCG@5: 0.8633 ± 0.0000
  mAP@5: 0.9732 ± 0.0000
  Recall@10: 0.7894 ± 0.0000
  NDCG@10: 0.9851 ± 0.0000
  mAP@10: 1.1148 ± 0.0000
  Recall@15: 0.8264 ± 0.0000
  NDCG@15: 1.0510 ± 0.0000
  mAP@15: 1.1845 ± 0.0000
Saved index C α=0.5


## Cell 13 — Save All Metrics & Print Summary Table

In [13]:
# Save all results
metrics_path = OUTPUT_DIR / "all_metrics.json"
json.dump(all_results, open(metrics_path, "w"), indent=2)
print(f"Saved all metrics → {metrics_path}")

# Pretty print comparison table
print("\n" + "="*70)
print(f"{'Condition':<35} {'R@5':>7} {'R@10':>7} {'mAP@5':>8} {'mAP@10':>8} {'NDCG@10':>9}")
print("="*70)
for cond, results in all_results.items():
    r5   = results.get('Recall@5',  {}).get('mean', 0)
    r10  = results.get('Recall@10', {}).get('mean', 0)
    m5   = results.get('mAP@5',     {}).get('mean', 0)
    m10  = results.get('mAP@10',    {}).get('mean', 0)
    n10  = results.get('NDCG@10',   {}).get('mean', 0)
    print(f"{cond:<35} {r5:>7.4f} {r10:>7.4f} {m5:>8.4f} {m10:>8.4f} {n10:>9.4f}")
print("="*70)

Saved all metrics → /kaggle/working/all_metrics.json

Condition                               R@5    R@10    mAP@5   mAP@10   NDCG@10
A_vision_only_alpha1                 0.5132  0.5771   0.6287   0.7083    0.6489
B_frozen_clip_alpha07                0.3292  0.4096   0.3192   0.3732    0.3745
B_frozen_clip_alpha05                0.0148  0.0740   0.0089   0.0213    0.0347
C_finetuned_clip_alpha07             0.8304  0.8775   1.4518   1.6673    1.3397
C_finetuned_clip_alpha05             0.7122  0.7894   0.9732   1.1148    0.9851


## Cell 14 — Online Query Function (for Streamlit / Batch Eval)
Use this function in your Streamlit demo and batch evaluation script.

In [14]:
# from ultralytics import YOLO as UltralyticsYOLO

# # Load YOLO (same model as BLIP notebook's online crop)
# yolo_model = UltralyticsYOLO(str(YOLO_PT))


# def online_crop(img_path_or_pil, yolo=yolo_model, pad=PAD):
#     """
#     Crop a query image at runtime using YOLO.
#     Used for query images that have no ground truth bbox annotation.
#     """
#     if isinstance(img_path_or_pil, (str, Path)):
#         pil_img = Image.open(img_path_or_pil).convert("RGB")
#     else:
#         pil_img = img_path_or_pil.convert("RGB")

#     result = yolo.predict(source=pil_img, conf=0.25, verbose=False)[0]
#     if result.boxes and len(result.boxes):
#         best = max(result.boxes, key=lambda b: float(b.conf[0]))
#         x1, y1, x2, y2 = best.xyxy[0].cpu().numpy().astype(int)
#         W, H = pil_img.size
#         px = int((x2 - x1) * pad); py = int((y2 - y1) * pad)
#         return pil_img.crop((max(0, x1-px), max(0, y1-py), min(W, x2+px), min(H, y2+py)))
#     return pil_img   # fallback: no detection


# @torch.no_grad()
# def retrieve_top_k(query_img_pil, clip_model, tokenizer, clip_preprocess,
#                    hnsw_index, gallery_ids, gallery_names, alpha, k=10):
#     """
#     Given a PIL query image (already YOLO-cropped), retrieve top-K gallery items.

#     Returns list of dicts: [{item_id, image_name, score}, ...]
#     """
#     clip_model.eval()
#     img_tensor = clip_preprocess(query_img_pil).unsqueeze(0).to(DEVICE)
#     img_emb = F.normalize(clip_model.encode_image(img_tensor).float(), dim=-1)

#     # For query we only have the image (no caption at retrieval time)
#     # Use alpha=1.0 for the query embedding regardless of gallery alpha
#     query_emb = img_emb.cpu().numpy().astype(np.float32)

#     labels, distances = hnsw_index.knn_query(query_emb, k=k)
#     results = []
#     for pos, dist in zip(labels[0], distances[0]):
#         results.append({
#             "item_id":    gallery_ids[pos],
#             "image_name": gallery_names[pos],
#             "score":      float(1 - dist),   # hnswlib cosine distance -> similarity
#         })
#     return results


# print("YOLO loaded. Online query functions ready.")
# print("Use online_crop() + retrieve_top_k() in Streamlit demo and batch eval script.")